In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.gold.fct_demo")
spark.sql("CREATE TABLE workspace.gold.fct_demo AS SELECT * FROM workspace.gold.fct_order_items")

display(spark.sql("""
SELECT COUNT(*) AS items, ROUND(SUM(profit), 0) AS total_profit
FROM workspace.gold.fct_demo
"""))


In [0]:
spark.sql("UPDATE workspace.gold.fct_demo SET profit = 0 WHERE shipping_delay_days > 2")

display(spark.sql("DESCRIBE HISTORY workspace.gold.fct_demo").select("version", "timestamp", "operation"))

In [0]:
spark.sql("RESTORE TABLE workspace.gold.fct_demo TO VERSION AS OF 0")

display(spark.sql("""
SELECT 'original table' AS stage, COUNT(*) AS items, ROUND(SUM(profit), 0) AS total_profit
FROM workspace.gold.fct_order_items
UNION ALL
SELECT 'demo: before damage (version 0)', COUNT(*), ROUND(SUM(profit), 0)
FROM workspace.gold.fct_demo VERSION AS OF 0
UNION ALL
SELECT 'demo: after damage (version 1)', COUNT(*), ROUND(SUM(profit), 0)
FROM workspace.gold.fct_demo VERSION AS OF 1
UNION ALL
SELECT 'demo: after RESTORE (current)', COUNT(*), ROUND(SUM(profit), 0)
FROM workspace.gold.fct_demo
"""))

In [0]:
from pyspark.sql import functions as F

base = spark.table("workspace.gold.fct_demo").orderBy("order_item_key")
upd = base.limit(3).withColumn("profit", (F.col("profit") + 100).cast("decimal(18,2)"))
new = base.limit(2).withColumn("order_item_key", F.col("order_item_key") + 10000000)
upd.unionByName(new).createOrReplaceTempView("fct_updates")

In [0]:
display(spark.sql("""
MERGE INTO workspace.gold.fct_demo AS t
USING fct_updates AS s
ON t.order_item_key = s.order_item_key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
"""))

In [0]:
display(spark.sql("SELECT COUNT(*) AS items FROM workspace.gold.fct_demo"))

In [0]:
display(spark.sql("DESCRIBE DETAIL workspace.gold.fct_demo").select("numFiles", "sizeInBytes"))

In [0]:
display(spark.sql("OPTIMIZE workspace.gold.fct_demo"))

In [0]:
display(spark.sql("DESCRIBE DETAIL workspace.gold.fct_demo").select("numFiles", "sizeInBytes"))

In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.gold.fct_demo")
spark.sql("DROP VIEW IF EXISTS fct_updates")